# COVFIL Round-Trip Tests

Tests for the rewritten `read_covfil` parser and the new `write_covfil` writer.

- **Test 1** — MF33 read: verify CovMat is returned with correct metadata
- **Test 2** — MF33 round-trip: read → write → read back → compare
- **Test 3** — MF34 read: verify MF34CovMat is returned with correct metadata
- **Test 4** — MF34 round-trip: read → write → read back → compare

In [1]:
import tempfile
import os
import numpy as np

from kika.cov import read_covfil, write_covfil
from kika.cov.covmat import CovMat
from kika.cov.mf34_covmat import MF34CovMat

MF33_FILE = r'E:\OneDrive\Share\Juan\260560_40.02.xs.gendf'
MF34_FILE = r'E:\OneDrive\Share\Juan\26056_mf34_02.gendf'

## Test 0 — Spot-Check Against Raw File Values

Manually extracted reference values from the raw GENDF files
to verify the parser reads them correctly — independent of round-trip.

In [2]:
mf33 = read_covfil(MF33_FILE)
mf34 = read_covfil(MF34_FILE)

# ── MF33: Energy grid (hand-read from lines 4-13 of the raw file) ──
# Line 4: 1.000000-5  4.000000-3  1.000000-2  2.530000-2  4.000000-2  5.000000-2
# Line 13: 4.304000+6  6.434000+6  2.000000+7
assert np.isclose(mf33.energy_grid[0], 1e-5), f'E[0]={mf33.energy_grid[0]}'
assert np.isclose(mf33.energy_grid[1], 4e-3), f'E[1]={mf33.energy_grid[1]}'
assert np.isclose(mf33.energy_grid[2], 1e-2), f'E[2]={mf33.energy_grid[2]}'
assert np.isclose(mf33.energy_grid[-1], 2e7), f'E[-1]={mf33.energy_grid[-1]}'
assert np.isclose(mf33.energy_grid[-2], 6.434e6), f'E[-2]={mf33.energy_grid[-2]}'
assert np.isclose(mf33.energy_grid[-3], 4.304e6), f'E[-3]={mf33.energy_grid[-3]}'
print('Energy grid spot-check: OK')

# ── MF33: Cross sections (hand-read from MF3 section) ──
# MT=1 first value (line 17): 2.223521+1 = 22.23521
# MT=1 last value  (line 26): 3.373539+0 = 3.373539
xs1 = mf33.cross_sections[(26056, 1)]
assert np.isclose(xs1[0], 22.23521, rtol=1e-6), f'XS(1)[0]={xs1[0]}'
assert np.isclose(xs1[-1], 3.373539, rtol=1e-6), f'XS(1)[-1]={xs1[-1]}'

# MT=2 first value (line 29): 1.347204+1 = 13.47204
# MT=102 first value (line 77): 8.763172+0 = 8.763172
xs2 = mf33.cross_sections[(26056, 2)]
assert np.isclose(xs2[0], 13.47204, rtol=1e-6), f'XS(2)[0]={xs2[0]}'
xs102 = mf33.cross_sections[(26056, 102)]
assert np.isclose(xs102[0], 8.763172, rtol=1e-6), f'XS(102)[0]={xs102[0]}'
print('Cross-section spot-check: OK')

# ── MF33: Covariance matrix values (hand-read from MF33 MT=1 section) ──
# Matrix (1,1) — row 1, col 1 (line 104): 4.761941-4
# Matrix (1,1) — row 2, col 1 (line 114): 3.555683-4
# Matrix (1,1) — row 1, col 2 (line 114): 3.555683-4  (symmetric!)
# Matrix (1,1) — row 3, col 1 (line 124): 2.816217-4
idx_mt1 = mf33.reaction_rows.index(1)
m1 = mf33.matrices[idx_mt1]
assert np.isclose(m1[0, 0], 4.761941e-4, rtol=1e-5), f'm1[0,0]={m1[0,0]}'
assert np.isclose(m1[1, 0], 3.555683e-4, rtol=1e-5), f'm1[1,0]={m1[1,0]}'
assert np.isclose(m1[0, 1], 3.555683e-4, rtol=1e-5), f'm1[0,1]={m1[0,1]}'  # symmetric
assert np.isclose(m1[2, 0], 2.816217e-4, rtol=1e-5), f'm1[2,0]={m1[2,0]}'

# Matrix (103,103) — sparse: only rows 54-56 have data
# Row 54, cols 54-56 (line 1752): 2.416547-7, 3.615148-5, 5.349719-6
# Row 56, cols 54-56 (line 1756): 5.349719-6, 1.297177-3, 5.009216-4
idx_mt103 = mf33.reaction_rows.index(103)
m103 = mf33.matrices[idx_mt103]
assert np.isclose(m103[53, 53], 2.416547e-7, rtol=1e-5), f'm103[53,53]={m103[53,53]}'
assert np.isclose(m103[53, 54], 3.615148e-5, rtol=1e-5), f'm103[53,54]={m103[53,54]}'
assert np.isclose(m103[55, 55], 5.009216e-4, rtol=1e-5), f'm103[55,55]={m103[55,55]}'
# Zero region should be zero
assert m103[0, 0] == 0.0, f'Expected zero at m103[0,0], got {m103[0,0]}'
assert m103[10, 10] == 0.0, f'Expected zero at m103[10,10], got {m103[10,10]}'
print('MF33 covariance matrix spot-check: OK')

# ── MF34: Matrix values (hand-read from MF34 MT=251 section) ──
# Row 1, col 1 (line 32): 9.072000-2
# Row 54, col 54 (line 502): 9.374141-5
# Row 56, col 56 (line 506): 2.321248-4
m34 = mf34.matrices[0]
assert np.isclose(m34[0, 0], 9.072e-2, rtol=1e-5), f'm34[0,0]={m34[0,0]}'
assert np.isclose(m34[53, 53], 9.374141e-5, rtol=1e-5), f'm34[53,53]={m34[53,53]}'
assert np.isclose(m34[55, 55], 2.321248e-4, rtol=1e-5), f'm34[55,55]={m34[55,55]}'
# Row 56, col 54 (line 506): 6.947195-6
assert np.isclose(m34[55, 53], 6.947195e-6, rtol=1e-5), f'm34[55,53]={m34[55,53]}'
print('MF34 covariance matrix spot-check: OK')

print()
print('All spot-check assertions passed — parser matches raw file values.')

Energy grid spot-check: OK
Cross-section spot-check: OK
MF33 covariance matrix spot-check: OK
MF34 covariance matrix spot-check: OK

All spot-check assertions passed — parser matches raw file values.


## Test 1 — MF33 Read

In [3]:
# Re-use mf33 from Test 0
assert isinstance(mf33, CovMat), f'Expected CovMat, got {type(mf33).__name__}'
assert mf33.num_groups == 56, f'Expected 56 groups, got {mf33.num_groups}'
assert mf33.energy_grid is not None, 'Energy grid is None'
assert len(mf33.energy_grid) == 57, f'Expected 57 energy boundaries, got {len(mf33.energy_grid)}'
assert mf33.num_matrices > 0, 'No matrices found'
assert len(mf33.cross_sections) > 0, 'No cross sections found'

print(f'MF33 read: {mf33.num_matrices} matrices, {mf33.num_groups} groups')
print(f'  Energy grid: {mf33.energy_grid[0]:.4e} .. {mf33.energy_grid[-1]:.4e} eV')
print(f'  Cross sections: {len(mf33.cross_sections)} entries')
print(f'  Reactions: {list(zip(mf33.reaction_rows, mf33.reaction_cols))}')

# Verify matrix shapes
for i, m in enumerate(mf33.matrices):
    assert m.shape == (56, 56), f'Matrix {i} has wrong shape: {m.shape}'

# Cross-section vector lengths
for key, xs in mf33.cross_sections.items():
    assert len(xs) == 56, f'XS {key} has wrong length: {len(xs)}'

print('\nAll MF33 read assertions passed.')

MF33 read: 7 matrices, 56 groups
  Energy grid: 1.0000e-05 .. 2.0000e+07 eV
  Cross sections: 7 entries
  Reactions: [(1, 1), (2, 2), (4, 4), (5, 5), (16, 16), (102, 102), (103, 103)]

All MF33 read assertions passed.


## Test 2 — MF33 Round-Trip

In [4]:
# Write to temp file
tmp33 = tempfile.mktemp(suffix='.gendf')
write_covfil(mf33, tmp33, tape_label='MF33 round-trip test', temperature=293.6)

# Read back
mf33_rt = read_covfil(tmp33)

# Compare metadata
assert isinstance(mf33_rt, CovMat)
assert mf33_rt.num_groups == mf33.num_groups
assert mf33_rt.num_matrices == mf33.num_matrices, \
    f'Matrix count mismatch: {mf33_rt.num_matrices} vs {mf33.num_matrices}'

# Compare energy grid
assert np.allclose(mf33.energy_grid, mf33_rt.energy_grid, atol=1e-6), \
    'Energy grid mismatch'

# Compare matrices
for i in range(mf33.num_matrices):
    max_diff = np.max(np.abs(mf33.matrices[i] - mf33_rt.matrices[i]))
    assert np.allclose(mf33.matrices[i], mf33_rt.matrices[i], atol=1e-6), \
        f'Matrix {i} mismatch: max_diff={max_diff:.2e}'
    assert mf33.reaction_rows[i] == mf33_rt.reaction_rows[i]
    assert mf33.reaction_cols[i] == mf33_rt.reaction_cols[i]
    assert mf33.isotope_rows[i] == mf33_rt.isotope_rows[i]
    assert mf33.isotope_cols[i] == mf33_rt.isotope_cols[i]

# Compare cross sections
for key in mf33.cross_sections:
    assert key in mf33_rt.cross_sections, f'Missing XS key: {key}'
    assert np.allclose(mf33.cross_sections[key], mf33_rt.cross_sections[key], atol=1e-6), \
        f'XS mismatch for {key}'

os.unlink(tmp33)
print(f'MF33 round-trip: all {mf33.num_matrices} matrices match within atol=1e-6')
print('All MF33 round-trip assertions passed.')

MF33 round-trip: all 7 matrices match within atol=1e-6
All MF33 round-trip assertions passed.


## Test 3 — MF34 Read

In [5]:
mf34 = read_covfil(MF34_FILE)

assert isinstance(mf34, MF34CovMat), f'Expected MF34CovMat, got {type(mf34).__name__}'
assert mf34.num_matrices > 0, 'No matrices found'

print(f'MF34 read: {mf34.num_matrices} matrices')
for i in range(mf34.num_matrices):
    print(f'  [{i}] MT={mf34.reaction_rows[i]}, L={mf34.l_rows[i]}, L1={mf34.l_cols[i]}, '
          f'shape={mf34.matrices[i].shape}, '
          f'E_grid={len(mf34.energy_grids[i])} pts')

# Check Legendre orders are positive
for i in range(mf34.num_matrices):
    assert mf34.l_rows[i] >= 0, f'Negative L_row at index {i}'
    assert mf34.l_cols[i] >= 0, f'Negative L_col at index {i}'

print('\nAll MF34 read assertions passed.')

MF34 read: 1 matrices
  [0] MT=251, L=1, L1=1, shape=(56, 56), E_grid=57 pts

All MF34 read assertions passed.


## Test 4 — MF34 Round-Trip

In [6]:
# Write to temp file
tmp34 = tempfile.mktemp(suffix='.gendf')
write_covfil(mf34, tmp34, tape_label='MF34 round-trip test', temperature=293.6)

# Read back
mf34_rt = read_covfil(tmp34)

# Compare
assert isinstance(mf34_rt, MF34CovMat)
assert mf34_rt.num_matrices == mf34.num_matrices, \
    f'Matrix count mismatch: {mf34_rt.num_matrices} vs {mf34.num_matrices}'

for i in range(mf34.num_matrices):
    max_diff = np.max(np.abs(mf34.matrices[i] - mf34_rt.matrices[i]))
    assert np.allclose(mf34.matrices[i], mf34_rt.matrices[i], atol=1e-6), \
        f'Matrix {i} mismatch: max_diff={max_diff:.2e}'
    assert mf34.l_rows[i] == mf34_rt.l_rows[i], f'L_row mismatch at {i}'
    assert mf34.l_cols[i] == mf34_rt.l_cols[i], f'L_col mismatch at {i}'
    assert mf34.reaction_rows[i] == mf34_rt.reaction_rows[i]
    assert mf34.reaction_cols[i] == mf34_rt.reaction_cols[i]
    assert np.allclose(mf34.energy_grids[i], mf34_rt.energy_grids[i], atol=1e-6)

os.unlink(tmp34)
print(f'MF34 round-trip: all {mf34.num_matrices} matrices match within atol=1e-6')
print('All MF34 round-trip assertions passed.')

MF34 round-trip: all 1 matrices match within atol=1e-6
All MF34 round-trip assertions passed.


## Test 5 — Class Method API

In [7]:
# CovMat.from_covfil
cm = CovMat.from_covfil(MF33_FILE)
assert isinstance(cm, CovMat)
assert cm.num_matrices == mf33.num_matrices

# MF34CovMat.from_covfil
m34 = MF34CovMat.from_covfil(MF34_FILE)
assert isinstance(m34, MF34CovMat)
assert m34.num_matrices == mf34.num_matrices

# Type guards
try:
    CovMat.from_covfil(MF34_FILE)
    assert False, 'Should have raised TypeError'
except TypeError:
    pass

try:
    MF34CovMat.from_covfil(MF33_FILE)
    assert False, 'Should have raised TypeError'
except TypeError:
    pass

# to_covfil via class method
tmp = tempfile.mktemp(suffix='.gendf')
cm.to_covfil(tmp, tape_label='Class method test')
cm2 = CovMat.from_covfil(tmp)
assert cm2.num_matrices == cm.num_matrices
os.unlink(tmp)

print('All class method API tests passed.')

All class method API tests passed.


## Test 6 — Old Parser vs New Parser

Inlines the old `read_covfil` logic (pre-rewrite) to compare its output
matrix-by-matrix against the new parser. This is the key backwards-compatibility test.

In [8]:
import re
from kika._constants import ENDF_MAT_TO_ZAID

def _map_mat_old(mat_str):
    try:
        mat_int = int(mat_str.strip())
    except ValueError:
        return mat_str
    return str(ENDF_MAT_TO_ZAID.get(mat_int, mat_int))

def read_covfil_old(file_path):
    """Exact copy of the old read_covfil implementation (pre-rewrite)."""
    dikt_cov = {'ISO_H': [], 'REAC_H': [],
                'ISO_V': [], 'REAC_V': [],
                'STD':    []}
    xs_dict = {}

    with open(file_path, 'r') as f:
        lines = f.readlines()

    iMAT1 = lines[2][66:70]
    group_nb = int(lines[2].split()[2])

    grep_data = False
    val_tot_nb = start_x_idx = start_y_idx = None
    vals = []
    energymesh = None

    i_line = 0
    while i_line < (len(lines) - 4):
        i_line += 1
        line = lines[i_line]

        splited_part = [line[i * 11:(i + 1) * 11].replace(' ', '')
                        for i in range(6)]
        splited_part = [x for x in splited_part if x]

        infos_part = line[66:]
        iMAT = infos_part[:4]
        iMF = str(int(infos_part[4:6]))
        iMT = str(int(infos_part[6:9]))

        if (iMAT, iMF) != ('0', '0') and iMT == '0':
            continue
        if iMAT != '0' and (iMF, iMT) == ('0', '0'):
            continue

        if iMAT != '0' and iMF == '1' and iMT == '451':
            i_line += 1
            line = lines[i_line]
            LIST_MF1451 = [line[i * 11:(i + 1) * 11].replace(' ', '')
                           for i in range(6)]
            LIST_MF1451 = [x for x in LIST_MF1451 if x]
            energymesh = []
            while len(energymesh) < int(LIST_MF1451[4]):
                i_line += 1
                line = lines[i_line]
                energylist = [line[i * 11:(i + 1) * 11].replace(' ', '')
                              for i in range(6)]
                energylist = [x for x in energylist if x]
                for energy in energylist:
                    if re.search('-', energy[-3:]):
                        valE = energy[:-3] + energy[-3:].split('-')[0] + 'E-' + energy[-3:].split('-')[1]
                        valE = float(valE)
                    elif re.search(r'\+', energy):
                        valE = energy[:-3] + energy[-3:].split('+')[0] + 'E+' + energy[-3:].split('+')[1]
                        valE = float(valE)
                    else:
                        valE = float(energy)
                    energymesh.append(valE)
            dikt_cov['ISO_H'].append('0')
            dikt_cov['REAC_H'].append('0')
            dikt_cov['ISO_V'].append('0')
            dikt_cov['REAC_V'].append('0')
            dikt_cov['STD'].append(energymesh)
            continue

        if iMAT != '0' and iMF == '3' and iMT != '0':
            crossSectionLine = []
            while len(crossSectionLine) < int(splited_part[4]):
                i_line += 1
                line = lines[i_line]
                LIST_MF3 = [line[i * 11:(i + 1) * 11].replace(' ', '')
                            for i in range(6)]
                LIST_MF3 = [x for x in LIST_MF3 if x]
                for xs_str in LIST_MF3:
                    if re.search('-', xs_str[-3:]):
                        valXS = xs_str[:-3] + xs_str[-3:].split('-')[0] + 'E-' + xs_str[-3:].split('-')[1]
                    elif re.search(r'\+', xs_str):
                        valXS = xs_str[:-3] + xs_str[-3:].split('+')[0] + 'E+' + xs_str[-3:].split('+')[1]
                    else:
                        valXS = xs_str
                    crossSectionLine.append(float(valXS))
            iso_num = _map_mat_old(iMAT)
            xs_dict[(int(iso_num), int(iMT))] = np.array(crossSectionLine)
            continue

        if len(splited_part) > 4 and splited_part[2] == '0' and splited_part[4] == '0':
            reac_2_id = splited_part[3]
            grep_data = True
            sub_mat = np.zeros((group_nb, group_nb))
            continue

        elif grep_data:
            if (val_tot_nb, start_x_idx, start_y_idx) == (None, None, None):
                val_tot_nb = int(splited_part[2])
                start_x_idx = int(splited_part[3]) - 1
                start_y_idx = int(splited_part[5]) - 1
                continue

            for val_str in splited_part:
                if re.search('-', val_str[-3:]):
                    val = val_str[:-3] + val_str[-3:].split('-')[0] + 'E-' + val_str[-3:].split('-')[1]
                elif re.search(r'\+', val_str):
                    val = val_str[:-3] + val_str[-3:].split('+')[0] + 'E+' + val_str[-3:].split('+')[1]
                else:
                    val = val_str
                vals.append(float(val))

            if len(vals) == val_tot_nb:
                sub_mat[start_y_idx][start_x_idx:start_x_idx + val_tot_nb] = vals
                val_tot_nb = start_x_idx = start_y_idx = None
                vals = []
                if (len(lines[i_line + 1].split()) < 3
                        or lines[i_line + 1].split()[2] == '0'):
                    if not np.isclose(sub_mat.sum(), 0.0):
                        dikt_cov['ISO_H'].append(_map_mat_old(iMAT))
                        dikt_cov['REAC_H'].append(iMT)
                        dikt_cov['ISO_V'].append(_map_mat_old(iMAT1))
                        dikt_cov['REAC_V'].append(reac_2_id)
                        dikt_cov['STD'].append(sub_mat.tolist())
                    grep_data = False
                continue

    # Build CovMat
    covmat_old = CovMat(num_groups=group_nb, energy_unit='eV')
    if (dikt_cov['STD']
            and isinstance(dikt_cov['STD'][0], list)
            and len(dikt_cov['STD'][0]) == group_nb + 1):
        covmat_old.energy_grid = dikt_cov['STD'][0]
        start_idx = 1
    elif energymesh is not None:
        covmat_old.energy_grid = energymesh
        start_idx = 0
    else:
        start_idx = 0

    for idx in range(start_idx, len(dikt_cov['STD'])):
        try:
            iso_h = int(str(dikt_cov['ISO_H'][idx]).strip())
            reac_h = int(str(dikt_cov['REAC_H'][idx]).strip())
            iso_v = int(str(dikt_cov['ISO_V'][idx]).strip())
            reac_v = int(str(dikt_cov['REAC_V'][idx]).strip())
            matrix = np.array(dikt_cov['STD'][idx])
            if matrix.shape == (group_nb, group_nb):
                covmat_old.add_matrix(iso_h, reac_h, iso_v, reac_v, matrix)
        except Exception:
            continue
    covmat_old.cross_sections.update(xs_dict)
    return covmat_old

# ── Run old parser and compare with new ──
old = read_covfil_old(MF33_FILE)
new = mf33  # already loaded

print(f'Old parser: {old.num_matrices} matrices, {len(old.cross_sections)} XS')
print(f'New parser: {new.num_matrices} matrices, {len(new.cross_sections)} XS')
print()

# Energy grids should match
assert np.allclose(old.energy_grid, new.energy_grid), 'Energy grid mismatch!'
print('Energy grids: MATCH')

# Cross sections should match
for key in old.cross_sections:
    assert key in new.cross_sections, f'New parser missing XS key {key}'
    assert np.allclose(old.cross_sections[key], new.cross_sections[key]), \
        f'XS mismatch for {key}'
print(f'Cross sections ({len(old.cross_sections)} entries): MATCH')

# Compare matrices — match by (iso_row, mt_row, iso_col, mt_col)
old_pairs = set(zip(old.isotope_rows, old.reaction_rows,
                     old.isotope_cols, old.reaction_cols))
new_pairs = set(zip(new.isotope_rows, new.reaction_rows,
                     new.isotope_cols, new.reaction_cols))

common = old_pairs & new_pairs
only_old = old_pairs - new_pairs
only_new = new_pairs - old_pairs

print(f'\nMatrix comparison:')
print(f'  Common pairs: {len(common)}')
if only_old:
    print(f'  Only in OLD: {only_old}')
if only_new:
    print(f'  Only in NEW: {only_new}')

# For each common pair, compare element-wise
for pair in sorted(common):
    iso_r, mt_r, iso_c, mt_c = pair
    old_idx = list(zip(old.isotope_rows, old.reaction_rows,
                       old.isotope_cols, old.reaction_cols)).index(pair)
    new_idx = list(zip(new.isotope_rows, new.reaction_rows,
                       new.isotope_cols, new.reaction_cols)).index(pair)
    m_old = old.matrices[old_idx]
    m_new = new.matrices[new_idx]
    max_diff = np.max(np.abs(m_old - m_new))
    match = np.allclose(m_old, m_new)
    status = 'MATCH' if match else f'DIFF (max={max_diff:.2e})'
    print(f'  MT=({mt_r},{mt_c}): {status}')
    assert match, f'Matrix mismatch for MT=({mt_r},{mt_c}), max_diff={max_diff}'

print()
print('Old vs New parser comparison: PASSED')

Old parser: 7 matrices, 7 XS
New parser: 7 matrices, 7 XS

Energy grids: MATCH
Cross sections (7 entries): MATCH

Matrix comparison:
  Common pairs: 7
  MT=(1,1): MATCH
  MT=(2,2): MATCH
  MT=(4,4): MATCH
  MT=(5,5): MATCH
  MT=(16,16): MATCH
  MT=(102,102): MATCH
  MT=(103,103): MATCH

Old vs New parser comparison: PASSED
